# Projector definitions and validation

This notebook validates the reproducible baseline projectors used by the SDP scaffold. It is not a substitute for the final published-convention audit; it exists to make the convention executable and testable.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
from deltawkrel.projectors import (
    ProcessDims, L_valid_bipartite, L_A_before_B, L_B_before_A,
    d_output, white_noise_process, random_hermitian, superoperator_matrix,
    trace_replace, AI, AO, BI, BO
)

dims = ProcessDims(2,2,2,2)
D = dims.hilbert_dim
print(dims, "D=", D, "d_O=", dims.d_O)

## Trace-and-replace sanity check
The replacement map `_S W` preserves trace and inserts a normalized identity on the replaced subsystem(s).

In [ ]:
W = random_hermitian(dims, seed=10).real
for systems in ([AO], [BO], [AI, AO], [AI, AO, BI, BO]):
    R = trace_replace(W, dims, systems)
    print(systems, "trace diff", abs(np.trace(R) - np.trace(W)))
    assert np.allclose(np.trace(R), np.trace(W), atol=1e-8)

## Idempotence tests
These are the minimum checks required before the projectors are used in an SDP scaffold.

In [ ]:
for name, P in [("L_V", L_valid_bipartite), ("L_AB", L_A_before_B), ("L_BA", L_B_before_A)]:
    PW = P(W, dims)
    err = np.linalg.norm(P(PW, dims) - PW)
    print(f"{name}: idempotence error = {err:.3e}")
    assert err < 1e-8

## Superoperator export
The matrices below are executable artifacts. They can be saved and used by the SDP code to impose `(I-P) vec(W)=0` constraints.

In [ ]:
export_dir = ROOT / "exports"
export_dir.mkdir(exist_ok=True)
for name, P in [("L_V", L_valid_bipartite), ("L_AB", L_A_before_B), ("L_BA", L_B_before_A)]:
    M = superoperator_matrix(P, dims)
    np.save(export_dir / f"{name}_superoperator_d2.npy", M)
    print(name, M.shape, "||P²-P||=", np.linalg.norm(M @ M - M))

## Status before submission
This notebook now validates and exports baseline projectors. Before formal submission, the exact convention must still be cross-checked against the cited equations and the ideal-switch benchmark notebook must reproduce a published robustness value.